In [5]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column, gridplot
from bokeh.models import ColumnDataSource, HoverTool, CustomJS, FactorRange
import colorsys
from bokeh.transform import dodge
import os

# Standard dimensions for all plots
width = 2000
height = 800

def generate_distinct_colors(n):
    """Generate n visually distinct colors optimized for data visualization."""
    distinct_colors = [
        '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#393b79', '#637939'
    ]
    
    if n > len(distinct_colors):
        golden_ratio_conjugate = 0.618033988749895
        current_hue = 0.33
        
        for i in range(n - len(distinct_colors)):
            current_hue = (current_hue + golden_ratio_conjugate) % 1
            saturation = 0.6 + (i % 3) * 0.1
            value = 0.9 - (i % 2) * 0.3
            
            rgb = colorsys.hsv_to_rgb(current_hue, saturation, value)
            hex_color = '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255),
                int(rgb[1] * 255),
                int(rgb[2] * 255)
            )
            distinct_colors.append(hex_color)
    
    return distinct_colors[:n]

def parse_valve_data(data_string):
    """Parse fixed-width formatted valve data."""
    colspecs = [
        (0, 3),    # Index
        (3, 12),   # CATEGORY
        (12, 22),  # CONFIDENCE
        (22, 72),  # DESCRIPTION
        (72, 87),  # END_CONNECTIONS
        (87, 96),  # END_FINISH
        (96, 102), # ID
        (102, 108),# LENGTH
        (108, 130),# MANUFACTURING_PROCESS
        (130, 144),# MATERIAL_GRADE
        (144, 158),# MATERIAL_NAME
        (158, 180),# MATERIAL_SPECIFICATION
        (180, 195),# MISCELLANEOUS
        (195, 208),# PRESSURE_CLASS
        (208, 223),# PRIMARY_SCHEDULE
        (223, 235),# PRIMARY_SIZE
        (235, 252),# REDUCING_SCHEDULE
        (252, 265),# REDUCING_SIZE
        (265, 271),# TRIM
        (271, 282),# TYPE
        (282, 291) # score
    ]
    
    try:
        df = pd.read_fwf(pd.StringIO(data_string), colspecs=colspecs)
        return df
    except:
        return pd.DataFrame(columns=['DESCRIPTION'])

def get_first_5_desc(top_25_str):
    """Extract and format first 5 descriptions from top 25 results."""
    try:
        x = parse_valve_data(top_25_str)
        first_5_values = x['DESCRIPTION'].head(5).tolist()
        formatted_desc = '<br>'.join([f"{i+1}. {desc.strip()}" for i, desc in enumerate(first_5_values)])
        return formatted_desc
    except:
        return "Error parsing descriptions"

def calculate_rank_stats(df):
    """Calculate percentage of results in different rank ranges."""
    total_records = len(df)
    stats = {
        'Top 5': (df['rank'] < 5).sum() / total_records * 100,
        'Top 25': (df['rank'] < 25).sum() / total_records * 100,
        'Top 50': (df['rank'] < 50).sum() / total_records * 100,
        'Top 100': (df['rank'] < 100).sum() / total_records * 100,
        'Top 150': (df['rank'] < 150).sum() / total_records * 100
    }
    return stats

def load_and_process_csv(filename):
    """Load and process a CSV file, filtering out rank 10000."""
    df = pd.read_csv(f"jan1_results/{filename}")
    df = df[df['rank'] != 10000].copy()
    if 'top_25' in df.columns:
        df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
    
    # Calculate rank statistics
    df['rank_stats'] = calculate_rank_stats(df)
    return df

def create_distribution_plot(data_list, column_name, title, x_label, colors, names):
    """Create a distribution plot using histogram for multiple datasets."""
    p = figure(width=width, height=height,
              title=title,
              x_axis_label=x_label,
              y_axis_label='Count',
              background_fill_color='#FFFFFF')
    
    for idx, (data, color, name) in enumerate(zip(data_list, colors, names)):
        hist, edges = np.histogram(data[column_name], bins=50)
        source = ColumnDataSource({
            'top': hist,
            'left': edges[:-1],
            'right': edges[1:],
            'name': [name] * len(hist)
        })
        
        p.quad(top='top', bottom=0, left='left', right='right',
               fill_color=color, line_color='white', alpha=0.5,
               hover_fill_color=color, hover_fill_alpha=0.7,
               source=source, legend_label=name)
    
    p.legend.click_policy = "hide"
    p.legend.location = "top_right"
    p.legend.background_fill_alpha = 0.7
    p.add_tools(HoverTool(tooltips=[
        ('Dataset', '@name'),
        ('Range', '@left{0.0} to @right{0.0}'),
        ('Count', '@top')
    ]))
    
    return p

def create_rank_distribution_bar_chart(datasets, colors):
    """Create a bar chart showing rank distribution statistics."""
    categories = ['Top 5', 'Top 25', 'Top 50', 'Top 100', 'Top 150']
    dataset_names = [d['name'] for d in datasets]
    
    data = {'categories': categories}
    for dataset in datasets:
        stats = calculate_rank_stats(dataset['data'])
        data[dataset['name']] = [stats[cat] for cat in categories]
    
    source = ColumnDataSource(data)
    
    p = figure(width=width, height=height,
              x_range=categories,
              title='Rank Distribution Statistics',
              x_axis_label='Rank Range',
              y_axis_label='Percentage of Results (%)',
              background_fill_color='#FFFFFF')
    
    bar_width = 0.8 / len(datasets)
    for idx, (name, color) in enumerate(zip(dataset_names, colors)):
        x_offset = (idx - len(datasets)/2 + 0.5) * bar_width
        p.vbar(x=dodge('categories', x_offset, range=p.x_range),
               top=name,
               width=bar_width,
               source=source,
               color=color,
               legend_label=name)
    
    p.legend.click_policy = "hide"
    p.legend.location = "top_right"
    p.legend.background_fill_alpha = 0.7
    
    p.xgrid.grid_line_color = None
    p.y_range.start = 0
    
    p.add_tools(HoverTool(tooltips=[
        ('Dataset', '$name'),
        ('Category', '@categories'),
        ('Percentage', '@$name{0.00}%')
    ]))
    
    return p

def create_visualization(csv_files):
    """Create Bokeh visualization for multiple CSV files with varied line styles and distributions."""
    datasets = []
    prior_rank_data = None
    
    # First load all datasets
    for file in csv_files:
        try:
            df = load_and_process_csv(file)
            datasets.append({
                'name': file.replace('.csv', ''),
                'data': df
            })
            if prior_rank_data is None and 'prior_rank' in df.columns:
                prior_rank_data = df[['i', 'prior_rank']].copy()
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    colors = generate_distinct_colors(len(datasets))
    line_dashes = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    # Create time series plots
    p1 = figure(width=width, height=height, 
               title='Vector Study Comparison - Rank (excluding rank 10000)',
               x_axis_label='Index',
               y_axis_label='Rank',
               background_fill_color='#FFFFFF')

    p2 = figure(width=width, height=height, 
               title='Vector Study Comparison - Total Time',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)',
               background_fill_color='#FFFFFF')
    
    sources = []
    
    # Add prior rank first if available
    if prior_rank_data is not None:
        source_prior = ColumnDataSource(prior_rank_data)
        p1.line('i', 'prior_rank', line_color='black',
                line_dash='dashed', legend_label='Prior Rank',
                source=source_prior, line_width=2)
    
    # Add all other datasets
    for idx, dataset in enumerate(datasets):
        source = ColumnDataSource(dataset['data'])
        sources.append(source)
        
        line_dash = line_dashes[idx % len(line_dashes)]
        
        p1.line('i', 'rank', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p1.circle('i', 'rank', size=6, color=colors[idx],
                 alpha=0.7, source=source)
        
        p2.line('i', 'totaltime', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p2.circle('i', 'totaltime', size=6, color=colors[idx],
                 alpha=0.7, source=source)
    
    # Create distribution plots
    p3 = create_distribution_plot(
        [d['data'] for d in datasets],
        'rank',
        'Rank Distribution',
        'Rank',
        colors,
        [d['name'] for d in datasets]
    )
    
    p4 = create_distribution_plot(
        [d['data'] for d in datasets],
        'totaltime',
        'Total Time Distribution',
        'Time (seconds)',
        colors,
        [d['name'] for d in datasets]
    )
    
    # Create rank distribution bar chart
    p5 = create_rank_distribution_bar_chart(datasets, colors)
    
    # Configure hover tool for time series plots
    hover_tooltips = [
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc'),
        ('Top 5 Results', '@top_5_desc{safe}')
    ]
    
    hover_tool = HoverTool(tooltips=hover_tooltips)
    
    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)
    
    # Configure plot properties
    for p in [p1, p2, p3, p4, p5]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_left"
        p.legend.background_fill_alpha = 0.7
        p.grid.grid_line_color = "#E0E0E0"
        p.grid.grid_line_alpha = 0.6

        if p in [p3, p4]:
            p.legend.location = "top_right"

    p2.x_range = p1.x_range
    
    # Sync selections between time series plots
    js_code = """
        const sources = cb_obj.tags;
        const main = cb_obj;
        for (let s of sources) {
            if (s !== main) {
                s.selected.indices = main.selected.indices;
            }
        }
    """
    
    for source in sources:
        source.tags = sources
        source.selected.js_on_change('indices',
            CustomJS(args=dict(sources=sources), code=js_code)
        )
    
    # Create layout with all plots
    layout = column(p1, p2, p3, p4, p5)
    
    output_file("vector_study_comparison.html")
    show(layout)

#### ----------------- Grab all .csv files, plot them.

folder_path = 'jan1_results'

csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]
create_visualization(csv_files)

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, CustomJS
import colorsys

# Standard dimensions for all plots
width = 2000
height = 800

def generate_distinct_colors(n):
    """Generate n visually distinct colors optimized for data visualization."""
    distinct_colors = [
        '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#393b79', '#637939'
    ]
    
    if n > len(distinct_colors):
        golden_ratio_conjugate = 0.618033988749895
        current_hue = 0.33
        
        for i in range(n - len(distinct_colors)):
            current_hue = (current_hue + golden_ratio_conjugate) % 1
            saturation = 0.6 + (i % 3) * 0.1
            value = 0.9 - (i % 2) * 0.3
            
            rgb = colorsys.hsv_to_rgb(current_hue, saturation, value)
            hex_color = '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255),
                int(rgb[1] * 255),
                int(rgb[2] * 255)
            )
            distinct_colors.append(hex_color)
    
    return distinct_colors[:n]

def parse_valve_data(data_string):
    """Parse fixed-width formatted valve data."""
    colspecs = [
        (0, 3),    # Index
        (3, 12),   # CATEGORY
        (12, 22),  # CONFIDENCE
        (22, 72),  # DESCRIPTION
        (72, 87),  # END_CONNECTIONS
        (87, 96),  # END_FINISH
        (96, 102), # ID
        (102, 108),# LENGTH
        (108, 130),# MANUFACTURING_PROCESS
        (130, 144),# MATERIAL_GRADE
        (144, 158),# MATERIAL_NAME
        (158, 180),# MATERIAL_SPECIFICATION
        (180, 195),# MISCELLANEOUS
        (195, 208),# PRESSURE_CLASS
        (208, 223),# PRIMARY_SCHEDULE
        (223, 235),# PRIMARY_SIZE
        (235, 252),# REDUCING_SCHEDULE
        (252, 265),# REDUCING_SIZE
        (265, 271),# TRIM
        (271, 282),# TYPE
        (282, 291) # score
    ]
    
    try:
        df = pd.read_fwf(pd.StringIO(data_string), colspecs=colspecs)
        return df
    except:
        return pd.DataFrame(columns=['DESCRIPTION'])

def get_first_5_desc(top_25_str):
    """Extract and format first 5 descriptions from top 25 results."""
    try:
        x = parse_valve_data(top_25_str)
        first_5_values = x['DESCRIPTION'].head(5).tolist()
        formatted_desc = '<br>'.join([f"{i+1}. {desc.strip()}" for i, desc in enumerate(first_5_values)])
        return formatted_desc
    except:
        return "Error parsing descriptions"

def load_and_process_csv(filename):
    """Load and process a CSV file, filtering out rank 10000."""
    df = pd.read_csv(f"jan1_results/{filename}")
    df = df[df['rank'] != 10000].copy()
    if 'top_25' in df.columns:
        df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
    return df

def create_distribution_plot(data_list, column_name, title, x_label, colors, names):
    """Create a distribution plot using histogram for multiple datasets."""
    p = figure(width=width, height=height,
              title=title,
              x_axis_label=x_label,
              y_axis_label='Count',
              background_fill_color='#FFFFFF')
    
    for idx, (data, color, name) in enumerate(zip(data_list, colors, names)):
        hist, edges = np.histogram(data[column_name], bins=50)
        source = ColumnDataSource({
            'top': hist,
            'left': edges[:-1],
            'right': edges[1:],
            'name': [name] * len(hist)
        })
        
        p.quad(top='top', bottom=0, left='left', right='right',
               fill_color=color, line_color='white', alpha=0.5,
               hover_fill_color=color, hover_fill_alpha=0.7,
               source=source, legend_label=name)
    
    p.legend.click_policy = "hide"
    p.legend.location = "top_right"
    p.legend.background_fill_alpha = 0.7
    p.add_tools(HoverTool(tooltips=[
        ('Dataset', '@name'),
        ('Range', '@left{0.0} to @right{0.0}'),
        ('Count', '@top')
    ]))
    
    return p

def create_visualization(csv_files):
    """Create Bokeh visualization for multiple CSV files with varied line styles and distributions."""
    datasets = []
    prior_rank_data = None
    
    # First load all datasets
    for file in csv_files:
        try:
            df = load_and_process_csv(file)
            datasets.append({
                'name': file.replace('.csv', ''),
                'data': df
            })
            if prior_rank_data is None and 'prior_rank' in df.columns:
                prior_rank_data = df[['i', 'prior_rank']].copy()
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    colors = generate_distinct_colors(len(datasets))
    line_dashes = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    # Create time series plots
    p1 = figure(width=width, height=height, 
               title='Vector Study Comparison - Rank (excluding rank 10000)',
               x_axis_label='Index',
               y_axis_label='Rank',
               background_fill_color='#FFFFFF')

    p2 = figure(width=width, height=height, 
               title='Vector Study Comparison - Total Time',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)',
               background_fill_color='#FFFFFF')
    
    sources = []
    
    # Add prior rank first if available
    if prior_rank_data is not None:
        source_prior = ColumnDataSource(prior_rank_data)
        p1.line('i', 'prior_rank', line_color='black',
                line_dash='dashed', legend_label='Prior Rank',
                source=source_prior, line_width=2)
    
    # Add all other datasets
    for idx, dataset in enumerate(datasets):
        source = ColumnDataSource(dataset['data'])
        sources.append(source)
        
        line_dash = line_dashes[idx % len(line_dashes)]
        
        p1.line('i', 'rank', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p1.circle('i', 'rank', size=6, color=colors[idx],
                 alpha=0.7, source=source)
        
        p2.line('i', 'totaltime', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p2.circle('i', 'totaltime', size=6, color=colors[idx],
                 alpha=0.7, source=source)
    
    # Create distribution plots
    p3 = create_distribution_plot(
        [d['data'] for d in datasets],
        'rank',
        'Rank Distribution',
        'Rank',
        colors,
        [d['name'] for d in datasets]
    )
    
    p4 = create_distribution_plot(
        [d['data'] for d in datasets],
        'totaltime',
        'Total Time Distribution',
        'Time (seconds)',
        colors,
        [d['name'] for d in datasets]
    )
    
    # Configure hover tool for time series plots
    hover_tooltips = [
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc')
    ]
    
    if any('top_5_desc' in dataset['data'].columns for dataset in datasets):
        hover_tooltips.append(('Top 5 Results', '@top_5_desc{safe}'))
    
    hover_tool = HoverTool(tooltips=hover_tooltips)
    
    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)
    
    # Configure plot properties
    for p in [p1, p2, p3, p4]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_left"
        p.legend.background_fill_alpha = 0.7
        p.grid.grid_line_color = "#E0E0E0"
        p.grid.grid_line_alpha = 0.6

        if p == p3 or p == p4:
            p.legend.location = "top_right"

    p2.x_range = p1.x_range
    
    # Sync selections between time series plots
    js_code = """
        const sources = cb_obj.tags;
        const main = cb_obj;
        for (let s of sources) {
            if (s !== main) {
                s.selected.indices = main.selected.indices;
            }
        }
    """
    
    for source in sources:
        source.tags = sources
        source.selected.js_on_change('indices',
            CustomJS(args=dict(sources=sources), code=js_code)
        )
    
    # Create vertical layout with all plots
    layout = column(p1, p2, p3, p4)
    
    output_file("vector_study_comparison.html")
    show(layout)

# List of CSV files to process
csv_files = [
    'VECTOR_SEARCH_QD_SIZE_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_ONLY_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_OPENAI_LARGE_Jan1.csv'
]

create_visualization(csv_files)

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, CustomJS
import colorsys

# Standard dimensions for all plots
width = 2000
height = 800

def generate_distinct_colors(n):
    """Generate n visually distinct colors optimized for data visualization."""
    distinct_colors = [
        '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#393b79', '#637939'
    ]
    
    if n > len(distinct_colors):
        golden_ratio_conjugate = 0.618033988749895
        current_hue = 0.33
        
        for i in range(n - len(distinct_colors)):
            current_hue = (current_hue + golden_ratio_conjugate) % 1
            saturation = 0.6 + (i % 3) * 0.1
            value = 0.9 - (i % 2) * 0.3
            
            rgb = colorsys.hsv_to_rgb(current_hue, saturation, value)
            hex_color = '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255),
                int(rgb[1] * 255),
                int(rgb[2] * 255)
            )
            distinct_colors.append(hex_color)
    
    return distinct_colors[:n]

def parse_valve_data(data_string):
    """Parse fixed-width formatted valve data."""
    colspecs = [
        (0, 3),    # Index
        (3, 12),   # CATEGORY
        (12, 22),  # CONFIDENCE
        (22, 72),  # DESCRIPTION
        (72, 87),  # END_CONNECTIONS
        (87, 96),  # END_FINISH
        (96, 102), # ID
        (102, 108),# LENGTH
        (108, 130),# MANUFACTURING_PROCESS
        (130, 144),# MATERIAL_GRADE
        (144, 158),# MATERIAL_NAME
        (158, 180),# MATERIAL_SPECIFICATION
        (180, 195),# MISCELLANEOUS
        (195, 208),# PRESSURE_CLASS
        (208, 223),# PRIMARY_SCHEDULE
        (223, 235),# PRIMARY_SIZE
        (235, 252),# REDUCING_SCHEDULE
        (252, 265),# REDUCING_SIZE
        (265, 271),# TRIM
        (271, 282),# TYPE
        (282, 291) # score
    ]
    
    try:
        df = pd.read_fwf(pd.StringIO(data_string), colspecs=colspecs)
        return df
    except:
        return pd.DataFrame(columns=['DESCRIPTION'])

def get_first_5_desc(top_25_str):
    """Extract and format first 5 descriptions from top 25 results."""
    try:
        x = parse_valve_data(top_25_str)
        first_5_values = x['DESCRIPTION'].head(5).tolist()
        formatted_desc = '<br>'.join([f"{i+1}. {desc.strip()}" for i, desc in enumerate(first_5_values)])
        return formatted_desc
    except:
        return "Error parsing descriptions"

def load_and_process_csv(filename):
    """Load and process a CSV file, filtering out rank 10000."""
    df = pd.read_csv(f"jan1_results/{filename}")
    df = df[df['rank'] != 10000].copy()
    if 'top_25' in df.columns:
        df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
    return df

def create_distribution_plot(data_list, column_name, title, x_label, colors, names):
    """Create a distribution plot using histogram for multiple datasets."""
    p = figure(width=width, height=height,
              title=title,
              x_axis_label=x_label,
              y_axis_label='Count',
              background_fill_color='#FFFFFF')
    
    for idx, (data, color, name) in enumerate(zip(data_list, colors, names)):
        hist, edges = np.histogram(data[column_name], bins=50)
        source = ColumnDataSource({
            'top': hist,
            'left': edges[:-1],
            'right': edges[1:],
            'name': [name] * len(hist)
        })
        
        p.quad(top='top', bottom=0, left='left', right='right',
               fill_color=color, line_color='white', alpha=0.5,
               hover_fill_color=color, hover_fill_alpha=0.7,
               source=source, legend_label=name)
    
    p.legend.click_policy = "hide"
    p.legend.location = "top_right"
    p.legend.background_fill_alpha = 0.7
    p.add_tools(HoverTool(tooltips=[
        ('Dataset', '@name'),
        ('Range', '@left{0.0} to @right{0.0}'),
        ('Count', '@top')
    ]))
    
    return p

def create_visualization(csv_files):
    """Create Bokeh visualization for multiple CSV files with varied line styles and distributions."""
    datasets = []
    prior_rank_data = None
    
    # First load all datasets
    for file in csv_files:
        try:
            df = load_and_process_csv(file)
            datasets.append({
                'name': file.replace('.csv', ''),
                'data': df
            })
            if prior_rank_data is None and 'prior_rank' in df.columns:
                prior_rank_data = df[['i', 'prior_rank']].copy()
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    colors = generate_distinct_colors(len(datasets))
    line_dashes = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    # Create time series plots
    p1 = figure(width=width, height=height, 
               title='Vector Study Comparison - Rank (excluding rank 10000)',
               x_axis_label='Index',
               y_axis_label='Rank',
               background_fill_color='#FFFFFF')

    p2 = figure(width=width, height=height, 
               title='Vector Study Comparison - Total Time',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)',
               background_fill_color='#FFFFFF')
    
    sources = []
    
    # Add prior rank first if available
    if prior_rank_data is not None:
        source_prior = ColumnDataSource(prior_rank_data)
        p1.line('i', 'prior_rank', line_color='black',
                line_dash='dashed', legend_label='Prior Rank',
                source=source_prior, line_width=2)
    
    # Add all other datasets
    for idx, dataset in enumerate(datasets):
        # Process the data to include top 5 descriptions
        df = dataset['data'].copy()
        if 'top_25' in df.columns:
            df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
        
        # Create the data source with the enhanced data
        source = ColumnDataSource(df)
        sources.append(source)
        
        line_dash = line_dashes[idx % len(line_dashes)]
        
        p1.line('i', 'rank', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p1.circle('i', 'rank', size=6, color=colors[idx],
                 alpha=0.7, source=source)
        
        p2.line('i', 'totaltime', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p2.circle('i', 'totaltime', size=6, color=colors[idx],
                 alpha=0.7, source=source)
    
    # Create distribution plots
    p3 = create_distribution_plot(
        [d['data'] for d in datasets],
        'rank',
        'Rank Distribution',
        'Rank',
        colors,
        [d['name'] for d in datasets]
    )
    
    p4 = create_distribution_plot(
        [d['data'] for d in datasets],
        'totaltime',
        'Total Time Distribution',
        'Time (seconds)',
        colors,
        [d['name'] for d in datasets]
    )
    
    # Configure hover tool for time series plots
    hover_tooltips = [
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc')
    ]
    
    if any('top_5_desc' in dataset['data'].columns for dataset in datasets):
        hover_tooltips.append(('Top 5 Results', '@top_5_desc{safe}'))
    
    hover_tool = HoverTool(tooltips=hover_tooltips)
    
    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)
    
    # Configure plot properties
    for p in [p1, p2, p3, p4]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_left"
        p.legend.background_fill_alpha = 0.7
        p.grid.grid_line_color = "#E0E0E0"
        p.grid.grid_line_alpha = 0.6
    
    p2.x_range = p1.x_range
    
    # Sync selections between time series plots
    js_code = """
        const sources = cb_obj.tags;
        const main = cb_obj;
        for (let s of sources) {
            if (s !== main) {
                s.selected.indices = main.selected.indices;
            }
        }
    """
    
    for source in sources:
        source.tags = sources
        source.selected.js_on_change('indices',
            CustomJS(args=dict(sources=sources), code=js_code)
        )
    
    # Create vertical layout with all plots
    layout = column(p1, p2, p3, p4)
    
    output_file("vector_study_comparison.html")
    show(layout)

# List of CSV files to process
csv_files = [
    'VECTOR_SEARCH_QD_SIZE_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_ONLY_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_OPENAI_LARGE_Jan1.csv'
]

create_visualization(csv_files)

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, CustomJS
import colorsys

width = 2000

def generate_distinct_colors(n):
    """Generate n visually distinct colors optimized for data visualization."""
    distinct_colors = [
        '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#393b79', '#637939'
    ]
    
    if n > len(distinct_colors):
        golden_ratio_conjugate = 0.618033988749895
        current_hue = 0.33
        
        for i in range(n - len(distinct_colors)):
            current_hue = (current_hue + golden_ratio_conjugate) % 1
            saturation = 0.6 + (i % 3) * 0.1
            value = 0.9 - (i % 2) * 0.3
            
            rgb = colorsys.hsv_to_rgb(current_hue, saturation, value)
            hex_color = '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255),
                int(rgb[1] * 255),
                int(rgb[2] * 255)
            )
            distinct_colors.append(hex_color)
    
    return distinct_colors[:n]

def parse_valve_data(data_string):
    """Parse fixed-width formatted valve data."""
    colspecs = [
        (0, 3),    # Index
        (3, 12),   # CATEGORY
        (12, 22),  # CONFIDENCE
        (22, 72),  # DESCRIPTION
        (72, 87),  # END_CONNECTIONS
        (87, 96),  # END_FINISH
        (96, 102), # ID
        (102, 108),# LENGTH
        (108, 130),# MANUFACTURING_PROCESS
        (130, 144),# MATERIAL_GRADE
        (144, 158),# MATERIAL_NAME
        (158, 180),# MATERIAL_SPECIFICATION
        (180, 195),# MISCELLANEOUS
        (195, 208),# PRESSURE_CLASS
        (208, 223),# PRIMARY_SCHEDULE
        (223, 235),# PRIMARY_SIZE
        (235, 252),# REDUCING_SCHEDULE
        (252, 265),# REDUCING_SIZE
        (265, 271),# TRIM
        (271, 282),# TYPE
        (282, 291) # score
    ]
    
    try:
        df = pd.read_fwf(pd.StringIO(data_string), colspecs=colspecs)
        return df
    except:
        return pd.DataFrame(columns=['DESCRIPTION'])

def get_first_5_desc(top_25_str):
    """Extract and format first 5 descriptions from top 25 results."""
    try:
        x = parse_valve_data(top_25_str)
        first_5_values = x['DESCRIPTION'].head(5).tolist()
        formatted_desc = '<br>'.join([f"{i+1}. {desc.strip()}" for i, desc in enumerate(first_5_values)])
        return formatted_desc
    except:
        return "Error parsing descriptions"

def load_and_process_csv(filename):
    """Load and process a CSV file, filtering out rank 10000."""
    df = pd.read_csv(f"jan1_results/{filename}")
    df = df[df['rank'] != 10000].copy()
    if 'top_25' in df.columns:
        df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
    return df

def create_visualization(csv_files):
    """Create Bokeh visualization for multiple CSV files with varied line styles."""
    datasets = []
    prior_rank_data = None
    
    # First load all datasets
    for file in csv_files:
        try:
            df = load_and_process_csv(file)
            datasets.append({
                'name': file.replace('.csv', ''),
                'data': df
            })
            # Store prior rank data from the first file that has it
            if prior_rank_data is None and 'prior_rank' in df.columns:
                prior_rank_data = df[['i', 'prior_rank']].copy()
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    colors = generate_distinct_colors(len(datasets))
    line_dashes = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    p1 = figure(width=width, height=600, 
               title='Vector Study Comparison - Rank (excluding rank 10000)',
               x_axis_label='Index',
               y_axis_label='Rank',
               background_fill_color='#FFFFFF')

    p2 = figure(width=width, height=600, 
               title='Vector Study Comparison - Total Time',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)',
               background_fill_color='#FFFFFF')
    
    sources = []
    
    # Add prior rank first if available
    if prior_rank_data is not None:
        source_prior = ColumnDataSource(prior_rank_data)
        p1.line('i', 'prior_rank', line_color='black',
                line_dash='dashed', legend_label='Prior Rank',
                source=source_prior, line_width=2)
    
    # Add all other datasets
    for idx, dataset in enumerate(datasets):
        source = ColumnDataSource(dataset['data'])
        sources.append(source)
        
        # Alternate line styles for better visibility
        line_dash = line_dashes[idx % len(line_dashes)]
        
        # Plot lines with varied styles
        p1.line('i', 'rank', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p1.circle('i', 'rank', size=6, color=colors[idx],
                 alpha=0.7, source=source)
        
        p2.line('i', 'totaltime', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p2.circle('i', 'totaltime', size=6, color=colors[idx],
                 alpha=0.7, source=source)
    
    # Configure hover tool with top 5 descriptions
    hover_tooltips = [
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc')
    ]
    
    if any('top_5_desc' in dataset['data'].columns for dataset in datasets):
        hover_tooltips.append(('Top 5 Results', '@top_5_desc{safe}'))
    
    hover_tool = HoverTool(tooltips=hover_tooltips)
    
    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)
    
    # Configure plot properties
    for p in [p1, p2]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_left"
        p.legend.background_fill_alpha = 0.7
        p.grid.grid_line_color = "#E0E0E0"
        p.grid.grid_line_alpha = 0.6
    
    p2.x_range = p1.x_range
    
    # Sync selections between plots
    js_code = """
        const sources = cb_obj.tags;
        const main = cb_obj;
        for (let s of sources) {
            if (s !== main) {
                s.selected.indices = main.selected.indices;
            }
        }
    """
    
    for source in sources:
        source.tags = sources
        source.selected.js_on_change('indices',
            CustomJS(args=dict(sources=sources), code=js_code)
        )
    
    layout = column(p1, p2)
    output_file("vector_study_comparison.html")
    show(layout)

# List of CSV files to process
csv_files = [
    'VECTOR_SEARCH_QD_SIZE_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_ONLY_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_OPENAI_LARGE_Jan1.csv'
]

create_visualization(csv_files)

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column, gridplot
from bokeh.models import ColumnDataSource, HoverTool, CustomJS
import colorsys

width = 2000
dist_width = width // 2  # Width for distribution plots

def generate_distinct_colors(n):
    """Generate n visually distinct colors optimized for data visualization."""
    distinct_colors = [
        '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#393b79', '#637939'
    ]
    
    if n > len(distinct_colors):
        golden_ratio_conjugate = 0.618033988749895
        current_hue = 0.33
        
        for i in range(n - len(distinct_colors)):
            current_hue = (current_hue + golden_ratio_conjugate) % 1
            saturation = 0.6 + (i % 3) * 0.1
            value = 0.9 - (i % 2) * 0.3
            
            rgb = colorsys.hsv_to_rgb(current_hue, saturation, value)
            hex_color = '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255),
                int(rgb[1] * 255),
                int(rgb[2] * 255)
            )
            distinct_colors.append(hex_color)
    
    return distinct_colors[:n]

def parse_valve_data(data_string):
    """Parse fixed-width formatted valve data."""
    colspecs = [
        (0, 3),    # Index
        (3, 12),   # CATEGORY
        (12, 22),  # CONFIDENCE
        (22, 72),  # DESCRIPTION
        (72, 87),  # END_CONNECTIONS
        (87, 96),  # END_FINISH
        (96, 102), # ID
        (102, 108),# LENGTH
        (108, 130),# MANUFACTURING_PROCESS
        (130, 144),# MATERIAL_GRADE
        (144, 158),# MATERIAL_NAME
        (158, 180),# MATERIAL_SPECIFICATION
        (180, 195),# MISCELLANEOUS
        (195, 208),# PRESSURE_CLASS
        (208, 223),# PRIMARY_SCHEDULE
        (223, 235),# PRIMARY_SIZE
        (235, 252),# REDUCING_SCHEDULE
        (252, 265),# REDUCING_SIZE
        (265, 271),# TRIM
        (271, 282),# TYPE
        (282, 291) # score
    ]
    
    try:
        df = pd.read_fwf(pd.StringIO(data_string), colspecs=colspecs)
        return df
    except:
        return pd.DataFrame(columns=['DESCRIPTION'])

def get_first_5_desc(top_25_str):
    """Extract and format first 5 descriptions from top 25 results."""
    try:
        x = parse_valve_data(top_25_str)
        first_5_values = x['DESCRIPTION'].head(5).tolist()
        formatted_desc = '<br>'.join([f"{i+1}. {desc.strip()}" for i, desc in enumerate(first_5_values)])
        return formatted_desc
    except:
        return "Error parsing descriptions"

def load_and_process_csv(filename):
    """Load and process a CSV file, filtering out rank 10000."""
    df = pd.read_csv(f"jan1_results/{filename}")
    df = df[df['rank'] != 10000].copy()
    if 'top_25' in df.columns:
        df['top_5_desc'] = df['top_25'].apply(get_first_5_desc)
    return df

def create_distribution_plot(data_list, column_name, title, x_label, colors, names):
    """Create a distribution plot using histogram for multiple datasets."""
    p = figure(width=dist_width, height=400,
              title=title,
              x_axis_label=x_label,
              y_axis_label='Count',
              background_fill_color='#FFFFFF')
    
    for idx, (data, color, name) in enumerate(zip(data_list, colors, names)):
        hist, edges = np.histogram(data[column_name], bins=50)
        source = ColumnDataSource({
            'top': hist,
            'left': edges[:-1],
            'right': edges[1:],
            'name': [name] * len(hist)
        })
        
        p.quad(top='top', bottom=0, left='left', right='right',
               fill_color=color, line_color='white', alpha=0.5,
               hover_fill_color=color, hover_fill_alpha=0.7,
               source=source, legend_label=name)
    
    p.legend.click_policy = "hide"
    p.legend.location = "top_right"
    p.legend.background_fill_alpha = 0.7
    p.add_tools(HoverTool(tooltips=[
        ('Dataset', '@name'),
        ('Range', '@left{0.0} to @right{0.0}'),
        ('Count', '@top')
    ]))
    
    return p

def create_visualization(csv_files):
    """Create Bokeh visualization for multiple CSV files with varied line styles and distributions."""
    datasets = []
    prior_rank_data = None
    
    # First load all datasets
    for file in csv_files:
        try:
            df = load_and_process_csv(file)
            datasets.append({
                'name': file.replace('.csv', ''),
                'data': df
            })
            if prior_rank_data is None and 'prior_rank' in df.columns:
                prior_rank_data = df[['i', 'prior_rank']].copy()
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    colors = generate_distinct_colors(len(datasets))
    line_dashes = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    # Create time series plots
    p1 = figure(width=width, height=600, 
               title='Vector Study Comparison - Rank (excluding rank 10000)',
               x_axis_label='Index',
               y_axis_label='Rank',
               background_fill_color='#FFFFFF')

    p2 = figure(width=width, height=600, 
               title='Vector Study Comparison - Total Time',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)',
               background_fill_color='#FFFFFF')
    
    sources = []
    
    # Add prior rank first if available
    if prior_rank_data is not None:
        source_prior = ColumnDataSource(prior_rank_data)
        p1.line('i', 'prior_rank', line_color='black',
                line_dash='dashed', legend_label='Prior Rank',
                source=source_prior, line_width=2)
    
    # Add all other datasets
    for idx, dataset in enumerate(datasets):
        source = ColumnDataSource(dataset['data'])
        sources.append(source)
        
        line_dash = line_dashes[idx % len(line_dashes)]
        
        p1.line('i', 'rank', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p1.circle('i', 'rank', size=6, color=colors[idx],
                 alpha=0.7, source=source)
        
        p2.line('i', 'totaltime', line_color=colors[idx],
                legend_label=dataset['name'],
                source=source, line_width=2.5,
                line_dash=line_dash)
        p2.circle('i', 'totaltime', size=6, color=colors[idx],
                 alpha=0.7, source=source)
    
    # Create distribution plots
    p3 = create_distribution_plot(
        [d['data'] for d in datasets],
        'rank',
        'Rank Distribution',
        'Rank',
        colors,
        [d['name'] for d in datasets]
    )
    
    p4 = create_distribution_plot(
        [d['data'] for d in datasets],
        'totaltime',
        'Total Time Distribution',
        'Time (seconds)',
        colors,
        [d['name'] for d in datasets]
    )
    
    # Configure hover tool for time series plots
    hover_tooltips = [
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc')
    ]
    
    if any('top_5_desc' in dataset['data'].columns for dataset in datasets):
        hover_tooltips.append(('Top 5 Results', '@top_5_desc{safe}'))
    
    hover_tool = HoverTool(tooltips=hover_tooltips)
    
    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)
    
    # Configure plot properties
    for p in [p1, p2]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_left"
        p.legend.background_fill_alpha = 0.7
        p.grid.grid_line_color = "#E0E0E0"
        p.grid.grid_line_alpha = 0.6
    
    p2.x_range = p1.x_range
    
    # Sync selections between time series plots
    js_code = """
        const sources = cb_obj.tags;
        const main = cb_obj;
        for (let s of sources) {
            if (s !== main) {
                s.selected.indices = main.selected.indices;
            }
        }
    """
    
    for source in sources:
        source.tags = sources
        source.selected.js_on_change('indices',
            CustomJS(args=dict(sources=sources), code=js_code)
        )
    
    # Create layout with all plots
    layout = column(
        p1, 
        p2,
        gridplot([[p3, p4]], toolbar_location='right')
    )
    
    output_file("vector_study_comparison.html")
    show(layout)

# List of CSV files to process
csv_files = [
    'VECTOR_SEARCH_QD_SIZE_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_ONLY_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_LARGE_Jan1.csv',
    'VECTOR_SEARCH_ONLY_OPENAI_SMALL_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_MULTILINGUAL E5 BASE_Jan1.csv',
    'VECTOR_SEARCH_QD_SIZE_OPENAI_LARGE_Jan1.csv'
]

create_visualization(csv_files)

In [ ]:
## this is good, shows 4 different plots

import pandas as pd
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, CustomJS

# Define the file groups and their labels
file_groups = {
    'Vector Search Only': [
        'VECTOR_SEARCH_ONLY_MULTILINGUAL E5 BASE_Jan1.csv',
        'VECTOR_SEARCH_ONLY_OPENAI_LARGE_Jan1.csv',
        'VECTOR_SEARCH_ONLY_OPENAI_SMALL_Jan1.csv'
    ],
    'Vector Search QD Size': [
        'VECTOR_SEARCH_QD_SIZE_MULTILINGUAL E5 BASE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_OPENAI_LARGE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_OPENAI_SMALL_Jan1.csv'
    ],
    'Vector Search QD Size Type Mat Sch': [
        'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv'
    ],
    'Vector Search QD Size PD Type Mat Sch': [
        'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_MULTILINGUAL E5 BASE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_LARGE_Jan1.csv',
        'VECTOR_SEARCH_QD_SIZE_PD_TYPE_MAT_SCH_OPENAI_SMALL_Jan1.csv'
    ]
}

def get_top_5_results(top_25_str):
    """Extract top 5 results from the top_25 string."""
    try:
        results = top_25_str.split('\n')[:5]  # Get first 5 lines
        formatted_results = '<br>'.join([f"{i+1}. {res.strip()}" for i, res in enumerate(results)])
        return formatted_results
    except:
        return "No results available"

def create_plot_for_group(group_name, files, output_suffix=''):
    # Read datasets and filter out rank 10000
    dataframes = []
    for file in files:
        df = pd.read_csv(r"jan1_results/"+file)
        df = df[df['rank'] != 10000].copy()  # Filter out rank 10000
        df['top_5'] = df['top_25'].apply(get_top_5_results)
        model_name = file.split('_')[-2] if 'OPENAI' in file else 'E5 BASE'
        df['model'] = model_name
        dataframes.append(df)

    # Create sources
    sources = [ColumnDataSource(df) for df in dataframes]
    source_prior = ColumnDataSource(dataframes[0][['i', 'prior_rank']])

    # Create figures
    p1 = figure(width=1200, height=600,
               title=f'{group_name} - Rank Comparison',
               x_axis_label='Index',
               y_axis_label='Rank')

    p2 = figure(width=1200, height=600,
               title=f'{group_name} - Total Time Comparison',
               x_axis_label='Index',
               y_axis_label='Total Time (seconds)')

    # Define colors for different models
    colors = {
        'E5 BASE': '#1f77b4',
        'LARGE': '#ff7f0e',
        'SMALL': '#2ca02c'
    }

    # Plot lines and circles
    for source, df in zip(sources, dataframes):
        color = colors[df['model'].iloc[0]]
        label = f"{df['model'].iloc[0]}"

        # Plot 1 - Rank
        p1.line('i', 'rank', line_color=color, legend_label=label,
                source=source, line_width=2)
        p1.circle('i', 'rank', size=8, color=color, alpha=0.5,
                 source=source, legend_label=label)

        # Plot 2 - Total Time
        p2.line('i', 'totaltime', line_color=color, legend_label=label,
                source=source, line_width=2)
        p2.circle('i', 'totaltime', size=8, color=color, alpha=0.5,
                 source=source, legend_label=label)

    # Add prior rank as dashed black line
    p1.line('i', 'prior_rank', line_color='black', legend_label='Prior Rank',
            source=source_prior, line_width=2, line_dash='dashed')
    p1.circle('i', 'prior_rank', size=8, color='black', alpha=0.5,
             source=source_prior, legend_label='Prior Rank')

    # Configure hover tool
    hover_tool = HoverTool(tooltips=[
        ('Index', '@i'),
        ('Rank', '@rank'),
        ('Total Time', '@totaltime{0.000} s'),
        ('RFQ', '@inc_rfq'),
        ('Ground Truth', '@gt_desc'),
        ('Top 5 Results', '@top_5{safe}')
    ])

    p1.add_tools(hover_tool)
    p2.add_tools(hover_tool)

    # Configure legends
    for p in [p1, p2]:
        p.legend.click_policy = "hide"
        p.legend.location = "top_right"

    # Link ranges
    p2.x_range = p1.x_range

    # Create layout
    layout = column(p1, p2)
    
    # Save to file
    output_file(f"vector_study_comparison_{output_suffix}.html")
    show(layout)

# Create plots for each group
for group_name, files in file_groups.items():
    output_suffix = group_name.lower().replace(' ', '_')
    create_plot_for_group(group_name, files, output_suffix)